In [1]:
import polars as pl
from pathlib import Path

In [2]:
BASE_PATH = Path.cwd().parent / "titanic-competition"
TRAIN_PATH = BASE_PATH / "train.csv"

In [3]:
# As we saw there are two approaches : lazy and eager
# One loads the data into ram(but still better than pandas), and stores the workflow to follow when loading data from path.

eager_df = pl.read_csv(TRAIN_PATH)
lazy_df = pl.scan_csv(TRAIN_PATH)

In [4]:
# Now let's peek at the head 
eager_df.head()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


In [5]:
# Look at the cleanliness and format .
# Compared to pandas a much more descriptive and informative peeking
# Also looking closely above , it first loads the shape of the slice and then loads the data.

In [6]:
# Now instead of info here we have scheme which showcases all the colnames and it's dtype immediately each in a tuple.
eager_df.schema

Schema([('PassengerId', Int64),
        ('Survived', Int64),
        ('Pclass', Int64),
        ('Name', String),
        ('Sex', String),
        ('Age', Float64),
        ('SibSp', Int64),
        ('Parch', Int64),
        ('Ticket', String),
        ('Fare', Float64),
        ('Cabin', String),
        ('Embarked', String)])

In [7]:
# Many functions have same name compared to pandas .
# The creators of Polars wanted the transition to feel familiar.
# The real difference begins when we start manipulating the data.

eager_df.head()
eager_df.shape
eager_df.columns
eager_df.dtypes


[Int64,
 Int64,
 Int64,
 String,
 String,
 Float64,
 Int64,
 Int64,
 String,
 Float64,
 String,
 String]

In [8]:
eager_df.sample(5)

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
158,0,3,"""Corn, Mr. Harry""","""male""",30.0,0,0,"""SOTON/OQ 392090""",8.05,null,"""S"""
168,0,3,"""Skoog, Mrs. William (Anna Bern…","""female""",45.0,1,4,"""347088""",27.9,null,"""S"""
121,0,2,"""Hickman, Mr. Stanley George""","""male""",21.0,2,0,"""S.O.C. 14879""",73.5,null,"""S"""
828,1,2,"""Mallet, Master. Andre""","""male""",1.0,0,2,"""S.C./PARIS 2079""",37.0042,null,"""C"""
55,0,1,"""Ostby, Mr. Engelhart Cornelius""","""male""",65.0,0,1,"""113509""",61.9792,"""B30""","""C"""


In [9]:
# We cannnot do boolean masking like pandas and we strictly need to follow filter function for effecient optimized code.

In [10]:
# With pl.col() we can create expression to filter out effeciently.
# NOTE :  This is not a boolean array as eager_df[eager_df['Sex'=='female']] but an expression
type(pl.col("Sex")) # We can clearly see that this is a type of expression 

polars.expr.expr.Expr

In [11]:
female_survivor = eager_df.filter(
                    pl.col("Sex") == "female",
                    pl.col("Survived") == 1
)

In [12]:
female_survivor

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
9,1,3,"""Johnson, Mrs. Oscar W (Elisabe…","""female""",27.0,0,2,"""347742""",11.1333,null,"""S"""
10,1,2,"""Nasser, Mrs. Nicholas (Adele A…","""female""",14.0,1,0,"""237736""",30.0708,null,"""C"""
…,…,…,…,…,…,…,…,…,…,…,…
875,1,2,"""Abelson, Mrs. Samuel (Hannah W…","""female""",28.0,1,0,"""P/PP 3381""",24.0,null,"""C"""
876,1,3,"""Najib, Miss. Adele Kiamie ""Jan…","""female""",15.0,0,0,"""2667""",7.225,null,"""C"""
880,1,1,"""Potter, Mrs. Thomas Jr (Lily A…","""female""",56.0,0,1,"""11767""",83.1583,"""C50""","""C"""


In [13]:
# Polars isn't built around DataFrames.It's built around expressions.

In [14]:
# Now we will dealt with with_columns()
# Why we need this ? - To Chain multiple expressions into a single expression so that we can create multiple new columns at one go .
# Let's take for example  : 
df = eager_df.with_columns(
    (
        pl.col("SibSp") +
        pl.col("Parch") +
        1
    ).alias("FamilySize"), # We add alias to give the expression result a destiny , without this is there is no home for the result. 
                           # We can also speicfy the same name as prexisting column as well to modify it's behaviour.

    (
        pl.col("Age") < 18
    ).alias("IsChild"),

    (
        pl.col("Fare") > 100
    ).alias("HighFare")
)
# It can literally compute everything at one go whereas in pandas this would be too hectic , to create new features one by one and would be completely ineffecient.
# Why ? Cause in pandas we would have had to load each column once then add each value of other columns and so on ....

# But with_columns let's us make new columns instantly with effeciency; 

In [15]:
df.head()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,IsChild,HighFare
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str,i64,bool,bool
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S""",2,false,false
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C""",2,false,false
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S""",1,false,false
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S""",2,false,false
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S""",1,false,false


In [16]:
# We can also modify the current columns at one go too.
# For example : 
new_age = df.with_columns(
    (pl.col("Age") + 1).alias("Age")
)

In [17]:
new_age.head() # COMPARE this with the dataframe above to check whether there was an actual change in tha "Age" column

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,IsChild,HighFare
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str,i64,bool,bool
1,0,3,"""Braund, Mr. Owen Harris""","""male""",23.0,1,0,"""A/5 21171""",7.25,null,"""S""",2,false,false
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",39.0,1,0,"""PC 17599""",71.2833,"""C85""","""C""",2,false,false
3,1,3,"""Heikkinen, Miss. Laina""","""female""",27.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S""",1,false,false
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",36.0,1,0,"""113803""",53.1,"""C123""","""S""",2,false,false
5,0,3,"""Allen, Mr. William Henry""","""male""",36.0,0,0,"""373450""",8.05,null,"""S""",1,false,false


In [18]:
# Now when we have two or more expression then what happens is that both expression use the original age group simultaenously
# Polars guarantees:Every expression in one with_columns() sees the same original snapshot of the DataFrame.
# take this below example : 
eager_df.with_columns(
    (pl.col("Age") + 1).alias("Age"),
    (pl.col("Age") > 18).alias("Adult")
)

# Now what this it does is  : 
# 1) Give the original "Age" to both the equations
# 2) Exp 1 updated the "Age" by adding 1 but as the Exp 2 has the raw "Age" as well the updated "Age " is never passed to form the "Adult"

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Adult
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str,bool
1,0,3,"""Braund, Mr. Owen Harris""","""male""",23.0,1,0,"""A/5 21171""",7.25,null,"""S""",true
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",39.0,1,0,"""PC 17599""",71.2833,"""C85""","""C""",true
3,1,3,"""Heikkinen, Miss. Laina""","""female""",27.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S""",true
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",36.0,1,0,"""113803""",53.1,"""C123""","""S""",true
5,0,3,"""Allen, Mr. William Henry""","""male""",36.0,0,0,"""373450""",8.05,null,"""S""",true
…,…,…,…,…,…,…,…,…,…,…,…,…
887,0,2,"""Montvila, Rev. Juozas""","""male""",28.0,0,0,"""211536""",13.0,null,"""S""",true
888,1,1,"""Graham, Miss. Margaret Edith""","""female""",20.0,0,0,"""112053""",30.0,"""B42""","""S""",true
889,0,3,"""Johnston, Miss. Catherine Hele…","""female""",null,1,2,"""W./C. 6607""",23.45,null,"""S""",null


In [19]:
# If we really wanted to bypass this we can use 
eager_df.with_columns(
    (pl.col("Age") + 1).alias("Age")
).with_columns(
        (pl.col("Age") > 18).alias("Adult")
)

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Adult
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str,bool
1,0,3,"""Braund, Mr. Owen Harris""","""male""",23.0,1,0,"""A/5 21171""",7.25,null,"""S""",true
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",39.0,1,0,"""PC 17599""",71.2833,"""C85""","""C""",true
3,1,3,"""Heikkinen, Miss. Laina""","""female""",27.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S""",true
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",36.0,1,0,"""113803""",53.1,"""C123""","""S""",true
5,0,3,"""Allen, Mr. William Henry""","""male""",36.0,0,0,"""373450""",8.05,null,"""S""",true
…,…,…,…,…,…,…,…,…,…,…,…,…
887,0,2,"""Montvila, Rev. Juozas""","""male""",28.0,0,0,"""211536""",13.0,null,"""S""",true
888,1,1,"""Graham, Miss. Margaret Edith""","""female""",20.0,0,0,"""112053""",30.0,"""B42""","""S""",true
889,0,3,"""Johnston, Miss. Catherine Hele…","""female""",null,1,2,"""W./C. 6607""",23.45,null,"""S""",null


In [20]:
# Feature Engineering using with_columns() : 
"""
FamilySize = SibSp + Parch + 1
IsChild = Age < 18
FarePerPerson = Fare / FamilySize
"""

df = eager_df.with_columns(
    (pl.col("SibSp") + pl.col("Parch") + 1).alias("FamilySize"),
    (pl.col("Age") < 18).alias("IsChild")).with_columns(
        (pl.col("Fare") / pl.col("FamilySize")).alias("FarePerPerson")
    )


In [21]:
df

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,IsChild,FarePerPerson
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str,i64,bool,f64
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S""",2,false,3.625
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C""",2,false,35.64165
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S""",1,false,7.925
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S""",2,false,26.55
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S""",1,false,8.05
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
887,0,2,"""Montvila, Rev. Juozas""","""male""",27.0,0,0,"""211536""",13.0,null,"""S""",1,false,13.0
888,1,1,"""Graham, Miss. Margaret Edith""","""female""",19.0,0,0,"""112053""",30.0,"""B42""","""S""",1,false,30.0
889,0,3,"""Johnston, Miss. Catherine Hele…","""female""",null,1,2,"""W./C. 6607""",23.45,null,"""S""",4,null,5.8625


In [22]:
# A better version would be to substitute the equation itself instead of nesting the with_columns() again
df = eager_df.with_columns(
    (pl.col("SibSp") + pl.col("Parch") + 1).alias("FamilySize"),

    (pl.col("Age") < 18).alias("IsChild"),

    (
        pl.col("Fare") /
        (pl.col("SibSp") + pl.col("Parch") + 1)
    ).alias("FarePerPerson")
)

In [27]:
# Q :  Find the number of survivors 
n_survivors = eager_df.select("Survived").filter(pl.col("Survived")==1).sum()
n_survivors = eager_df.select("Survived").sum()
n_survivors = (
    eager_df
    .filter(pl.col("Survived") == 1)
    .height
)
# Mutiple ways to do the same thing but the second one is the most optimum followed by 3 and 2
